# **Práctica 4:**
# **Paralelismo a nivel de hilos: Paralelización mediante OpenMP y programación asíncrona del análisis forense de manipulación de imágenes digitales**

## Miembros de equipo:
## *Nicolás Grima Hernández* y *Miguel Perez Alonso*

#### **Tarea 0.1 Entrenamiento previo OpenMP:**

**0.1.1. ¿Para qué sirve la variable chunk?**

La variable chunk determina la granularidad del reparto de carga en una construcción de bucle paralelo. En este código específico, con un CHUNKSIZE de 100, el runtime de OpenMP descompone el espacio de iteraciones en paquetes (bloques) de 100 unidades. Estos paquetes son las unidades mínimas de trabajo que se asignan a cada hilo del equipo (thread team), permitiendo un control fino sobre el balanceo de carga frente a la sobrecarga
de gestión.


**0.1.2 Explica completamente el pragma:**

*#pragma omp parallel shared(a,b,c,chunk) private(i)*

La directiva #pragma omp parallel define una región paralela, activando el modelo de ejecución Fork-Join. Cuando el hilo maestro encuentra esta directiva, crea un equipo de hilos que ejecutan concurrentemente el bloque de código asociado.

- ¿Por qué y para qué se usa *shared(a,b,c,chunk)* en este
programa?

*shared(a, b, c, chunk)* establece que estas variables residen en el espacio de memoria global accesible por todos los hilos.
Las variables a, b y c se definen como compartidas para permitir el acceso concurrente a las estructuras de datos de entrada y asegurar que los resultados calculados persistan en la memoria principal tras la finalización de la región paralela.
La variable chunk es compartida porque actúa como un parámetro de configuración de solo lectura. No requiere copias locales, ya que su valor permanece constante y es utilizado por todos los hilos en la lógica de planificación.

- ¿Por qué la variable i está etiquetada como private en el
pragma?

*private(i)* es una cláusula fundamental para la ejecución correcta del programa. Al declararla como privada, cada hilo dispone de su propia instancia de la variable i en su pila local (stack).
Si la variable i fuera compartida, se produciría una condición de carrera o race condition, ya que múltiples hilos intentarían modificar simultáneamente el mismo contador. Esto provocaría accesos incorrectos a los índices de los vectores, resultados corruptos y comportamientos indeterminados.



**0.1.3 ¿Para qué sirve *schedule*? ¿Qué otras posibilidades hay?**

Lo que hace la *schedule* es definir el algoritmo de asignación de iteraciones a los hilos de ejecución. En nuestro código, schedule(dynamic, chunk) implementa una planificación bajo demanda donde cada hilo solicita un nuevo bloque de tamaño chunk conforme termina el anterior. Esta cláusula se muestra como la estrategia óptima para mitigar el desequilibrio de carga (load imbalance) cuando las iteraciones tienen costes computacionales diferentes.

Alternativas posibles de planificación en OpenMP:

· *static* (estática): El espacio de iteraciones se divide y asigna de forma determinista antes de la ejecución. Minimiza la sobrecarga (overhead) de gestión, pero es sensible a desequilibrios de carga. 

· *guided* (guiada): Los hilos reciben bloques de tamaño decreciente. Combina la baja sobrecarga inicial con un refinamiento del balanceo al final del bucle.

**0.1.4. ¿Qué tiempos y otras medidas de rendimiento podemos medir en secciones de código paralelizadas con OpenMP?**

Para caracterizar el comportamiento de un código paralelizado, se emplean las siguientes métricas fundamentales:
- *Tiempo de ejecución (Wall-clock time)*: Representa el tiempo real total transcurrido desde el inicio hasta el fin del programa. En entornos OpenMP, se mide con precisión mediante la función omp_get_wtime(). Es la métrica base sobre la cual se calculan todas las demás.
- *Speed-up*: Es el cociente entre el tiempo de ejecución secuencial y el tiempo de ejecución paralelo con P hilos. Esta métrica cuantifica cuántas veces más rápido es el programa tras la optimización. Un Speed-up igual al número de hilos se considera una aceleración lineal ideal.
- *Eficiencia*: Es la relación entre el Speed-up obtenido y el número de hilos utilizados. Se expresa habitualmente en porcentaje y mide el grado de aprovechamiento de los recursos del sistema. Una eficiencia del 100% indicaría que no hay pérdida de rendimiento por gestión de hilos.
- *Escalabilidad*: Evalúa cómo responde el sistema ante el incremento de recursos. *Strong Scaling* (escalabilidad fuerte) analiza la capacidad de reducir el tiempo de respuesta para un problema de tamaño constante al aumentar el número de hilos; *Weak Scaling* (escalabilidad débil) evalúa la capacidad de mantener un tiempo de ejecución constante cuando el tamaño del problema aumenta en la misma proporción que el número de hilos.
- *Sobrecarga (Overhead)*: Se refiere al tiempo adicional que consume la CPU en tareas que no son de cálculo directo, como la creación de hilos, la sincronización en barreras, el balanceo de carga y la comunicación. Un elevado overhead limita directamente el Speed-up máximo alcanzable.

#### **Tarea 0.2: Entrenamiento previo std::async**

**0.2.1 ¿Para qué sirve el parámetro *std::launch::async?***

El parámetro *std::launch::async* es una política de lanzamiento que indica explícitamente al runtime de C++ que la tarea debe ejecutarse de forma asíncrona en un hilo nuevo y separado de forma inmediata. Al utilizar esta política, se garantiza el solapamiento de la ejecución de las funciones, permitiendo que tareas independientes progresen simultáneamente. En el código de ejemplo, esto permite que mientras una tarea está en estado de espera (sleep), la otra pueda estar utilizando ciclos de CPU o su propia cuenta atrás, reduciendo el tiempo total de ejecución.

En el código del ejemplo, gracias a *std::launch::async*, las tareas 1 y 2 se ejecutan simultáneamente. Por eso, aunque sumen 5 segundos de trabajo (2000ms + 3000ms), el programa terminará en aproximadamente 3 segundos (el tiempo de la tarea más larga).

**0.2.2 Calcula el tiempo que tarda el programa con *std::launch::async* y
*std::launch::deferred*. ¿A qué se debe la diferencia de tiempos?**

- *Tiempo con std::launch::async*: Aproximadamente 3000 ms. Dado que las tareas se ejecutan en paralelo, el tiempo total de respuesta está limitado por la tarea de mayor duración (la Tarea 2, de 3 segundos). 
- *Tiempo con std::launch::deferred*: Aproximadamente 5000 ms. En este modo, la ejecución no es paralela; las tareas se ejecutan de forma secuencial una tras otra en el hilo principal, sumando sus tiempos individuales (2000 ms + 3000 ms). 

La diferencia reside en la política de evaluación. Mientras que *async* fuerza la creación de hilos para la ejecución simultánea, *deferred* pospone la ejecución de la función hasta que se invoca explícitamente el método *.get()* o *.wait()*, realizando una llamada a función convencional (síncrona) en el mismo hilo.

**0.2.3 ¿Qué diferencia hay entre los métodos *wait* y *get* de *std::future*?**

Ambos métodos de *std::future* se utilizan para gestionar la finalización de una tarea asíncrona, pero tienen algunas diferencias clave: El método *wait()* bloquea la ejecución del hilo invocador hasta que la tarea asíncrona termine, sin devolver ningún resultado, por lo que se emplea principalmente con fines de sincronización cuando solo interesa saber que la ejecución ha finalizado; mientras que el método *get()* también bloquea hasta la finalización, pero además recupera y devuelve el valor calculado por la tarea asíncrona. Es importante destacar que *get()* solo puede llamarse una vez por cada objeto future, ya que al hacerlo se mueve el resultado, quedando este inaccesible para llamadas posteriores.

**0.2.4 ¿Qué ventajas ofrece *std::async* frente a *std::thread*?**

La principal ventaja de *std::async* es que ofrece un nivel de abstracción superior orientado a tareas en lugar de a hilos. Sus beneficios clave son: 
- *Gestión de resultados*: *std::async* permite devolver valores directamente mediante objetos future, mientras que *std::thread* requiere mecanismos manuales complejos como std::promise o variables globales. 
- *Propagación de excepciones*: Si la función asíncrona lanza una excepción, *std::async* la captura y la relanza en el hilo principal al llamar a *.get()*, facilitando la depuración. 
- *Gestión del ciclo de vida*: Con *std::thread*, el programador debe decidir manualmente si hacer *join()* o *detach()*; con *std::async*, el sistema gestiona la finalización de forma más automatizada y segura. 


#### **Tarea 0.3: Entrenamiento previo *std::vector***

**0.3.1: ¿Cuál de las dos formas de inicializar el vector y rellenarlo es más eficiente? ¿Por qué?**

La segunda forma (inicializar el vector con un tamaño predefinido: *std::vector<*float*> v2(10000)*) es significativamente más eficiente.
La causa principal radica en la gestión de la memoria dinámica. Cuando se utiliza *push_back()* en un vector sin tamaño definido, el contenedor realiza múltiples realocaciones conforme crece. Cada vez que el vector agota su capacidad actual, el sistema debe:
1. Reservar un nuevo bloque de memoria más grande (normalmente el doble del anterior). 
2. Copiar todos los elementos existentes a la nueva ubicación. 
3. Liberar la memoria antigua.

Este proceso genera un alto coste de CPU y fragmentación de memoria. Al pre-asignar el tamaño, se realiza una única reserva de memoria, eliminando el overhead de copia y permitiendo que el sistema optimice el acceso directo a las posiciones mediante el operador de índice.

**0.3.2: ¿Podría ocurrir algún problema al paralelizar los dos bucles for? ¿Por qué?**

La posibilidad de paralelización depende de la seguridad de hilo (thread safety) de las operaciones realizadas:
- *Primer bucle (push_back)*: No es seguro paralelizarlo. La función *push_back()* no es atómica ni thread-safe; modifica la estructura interna del vector (su tamaño y, potencialmente, su dirección de memoria). Si varios hilos intentan ejecutar *push_back()* simultáneamente, se produciría una condición de carrera crítica que corrompería los punteros internos del vector, provocando probablemente un fallo de segmentación (segmentation fault). 
- *Segundo bucle (v2[i])*: Es totalmente seguro paralelizarlo. Dado que la memoria ya ha sido reservada y el tamaño del vector es fijo, cada hilo trabaja sobre un índice i único y predecible. Al no existir solapamiento entre las posiciones de memoria a las que accede cada hilo (cada uno escribe en su propia "celda"), no hay conflictos de escritura ni dependencias de datos, permitiendo una ejecución paralela perfecta.

### Tarea 1: Paralelización del análisis forense de manipulación de imágenes digitales

#### **Tarea 1.1: Analiza el código e identifica los distintos procesos**

**1. Identificación de procesos**

Tras revisar el flujo de ejecución en main.cc, podemos identificar los siguientes procesos independientes que actúan sobre la imagen original:

1. *Carga de datos*: El proceso load_from_file actúa como el nodo raíz.
2. *Análisis SRM (3x3 y 5x5)*: Dos ejecuciones de filtrado de ruido de alta frecuencia. 
3. *Análisis ELA*: Proceso de análisis de error de nivel que implica recompresión. 
4. *Análisis DCT (Inverso y Directo)*: Dos transformaciones de frecuencia independientes. 
5. *Serialización*: Cinco procesos finales de escritura a disco (save_to_file).